# 01. 데이터 전처리 — KICOX core+revision → master → state/run/transition panel, PPI/EIS validation

이 노트북은 원자료(`data/raw/`)를 읽어 최종 분석 가능한 데이터를 만든다. 처리 로직은 전부
`src/*.py`에 있고, 이 노트북은 실행 순서와 핵심 확인만 담당한다(대형 함수를 노트북 안에서
반복 구현하지 않는다).

| 항목 | 내용 |
| --- | --- |
| 실행 방식 | 오프라인 전용. `data/raw/`는 읽기 전용이며 네트워크에 접근하지 않는다 |
| 참고기간(reference) | 2018Q1 ~ 2026Q2 (산단 전체 장기 흐름) |
| 본분석기간(main) | 2022Q1 ~ 2026Q2 (10개 업종 국면분석) — 근거는 5절 참고 |
| 업종 | 음식료·섬유의복·목재종이·석유화학·비금속·철강·기계·전기전자·운송장비·기타 |
| 핵심 산출물 | `data/processed/kicox/{changwon_industry_master, changwon_total_master, changwon_state_panel}.csv` |
| 검증 산출물 | `data/processed/ppi/ppi_validation_panel.csv`, `data/processed/eis/eis_validation_panel.csv` |

PPI/EIS/CCI는 KICOX core에 병합하지 않는다. PPI 업종별 매핑은 이번 단계에서 확정하지 않는다(보류).

## 0. Setup — 오프라인 강제, data/raw/ 쓰기 방지 가드

In [1]:
import os, sys, json, subprocess
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

DIR_PROC = ROOT / 'data' / 'processed' / 'kicox'
DIR_LOG = ROOT / 'logs' / 'preprocessing'

import build_changwon_master as bcm
_raw_snapshot_before = bcm._raw_mtimes()

# 자식 프로세스(subprocess) stdout도 utf-8로 고정한다 (Windows 콘솔 cp949 깨짐 방지)
_env = os.environ.copy()
_env['PYTHONIOENCODING'] = 'utf-8'

print('root    :', ROOT.name)
print('pandas  :', pd.__version__)
print('offline : True — build_changwon_master.py 에는 네트워크 호출 코드가 없다')
print('raw 파일 수:', len(_raw_snapshot_before))

root    : Changwon-Industry-Employment-Monitor
pandas  : 3.0.5
offline : True — build_changwon_master.py 에는 네트워크 호출 코드가 없다
raw 파일 수: 532


## 1~2. KICOX core + revision 로드, 3~4. master 구축·QA

`src/build_changwon_master.py`가 `data/raw/kicox/core/`(공공데이터포털 원자료)와
`data/raw/kicox/revision/`(KICOX 연간보정본)를 읽어 마스터를 만든다.
- X(비공개)는 0으로 치환하지 않고 NaN + masked=1로 남긴다.
- 보정본이 있는 분기는 보정본을 우선 사용한다(값이 없으면 포털 원자료 유지).
- `data/processed/kicox/_previous/`에 이전 산출물이 있으면 값 차이를 자동으로 비교해 로그로 남긴다.

In [2]:
res = subprocess.run([sys.executable, str(ROOT/'src'/'build_changwon_master.py')],
                     capture_output=True, text=True, encoding='utf-8', errors='replace',
                     env=_env, cwd=ROOT)
print('build_changwon_master.py exit code:', res.returncode)
print(res.stdout[-5000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0

build_changwon_master.py exit code: 0
실행 2026-09-08 11:34:33 | base=Changwon-Industry-Employment-Monitor | offline-only
[load] production       : 84건
[load] employment       : 33건
[load] op_rate          : 33건
[load] firms_in         : 33건
[load] firms_op         : 33건
[load] production_total : 83건
[load] employment_total : 33건
[load] firms_in_total   : 83건
[load] op_rate_total    : 83건

=== 원자료 구조 요약 ===
  production       : 84건 20180131~20260630 | 열수 [np.int64(13)]
  employment       : 33건 20180331~20260331 | 열수 [np.int64(14)]
  op_rate          : 33건 20180331~20260331 | 열수 [np.int64(13)]
  firms_in         : 33건 20180331~20260331 | 열수 [np.int64(14)]
  firms_op         : 33건 20180331~20260331 | 열수 [np.int64(14)]
  production_total : 83건 20180131~20260331 | 열수 [np.int64(6), np.int64(7)]
  employment_total : 33건 20180331~20260331 | 열수 [np.int64(7)]
  firms_in_total   : 83건 20180131~20260331 | 열수 [np.int64(7)]
  op_rate_total    : 83건 20180131~20260331 | 열수 [np.int64(7), np.int64(8)]

[

In [3]:
ind = pd.read_csv(DIR_PROC / 'changwon_industry_master.csv')
tot = pd.read_csv(DIR_PROC / 'changwon_total_master.csv')
latest = json.load(open(DIR_LOG / 'latest_points.json', encoding='utf-8'))

print(f'industry master : {len(ind)}행 ({ind.quarter.nunique()}분기 x {ind.industry.nunique()}업종)')
print(f'total master    : {len(tot)}행')
print(f'기간            : {ind.quarter.min()} ~ {ind.quarter.max()}')
print(f'2023Q4 production 결측(정상, X): {ind.loc[ind.quarter=="2023Q4","production"].isna().sum()}/10')

industry master : 340행 (34분기 x 10업종)
total master    : 34행
기간            : 2018Q1 ~ 2026Q2
2023Q4 production 결측(정상, X): 10/10


In [4]:
qa = subprocess.run([sys.executable, str(ROOT/'src'/'qa_master.py')],
                    capture_output=True, text=True, encoding='utf-8', errors='replace',
                    env=_env, cwd=ROOT)
print(qa.stdout[-6000:])
print('QA exit code:', qa.returncode, '(0 = FAIL 없음. WARN/INFO는 실패가 아니라 정상적 예외/확인사항)')
assert qa.returncode == 0, 'qa_master.py 에서 FAIL이 발생했습니다 — 파이프라인 문제로 취급'

QA 실행 2026-09-08 11:34:53 | base=Changwon-Industry-Employment-Monitor

A. 구조
  [PASS] industry master 행수 = quarter수(34) x industry수(10) = 340 (실제 340)
  [PASS] industry 10개 (실제 10)
  [PASS] total master 행수 = quarter수 (실제 34 vs quarter 34)
  [PASS] (quarter, industry) 중복 0건 (실제 0)
  [PASS] total quarter 중복 0건 (실제 0)
  [PASS] 업종명 집합 정확히 일치 (차이: 없음)
  [PASS] 전체 분기에서 업종 수 = 10 (위반 분기: 없음)
  [PASS] quarter 형식 YYYYQ[1-4] 전부 일치 (위반: 없음)
  범위 : 2018Q1 ~ 2026Q2 (연속 기대 34개, 실제 34개)
  [PASS] 2018Q1~2026Q2 연속, 빠진 분기 없음 (누락: 없음)

B. 데이터 타입
  [PASS] industry.production numeric dtype (실제 float64)
  [PASS] industry.employment numeric dtype (실제 float64)
  [PASS] industry.op_rate_official numeric dtype (실제 float64)
  [PASS] industry.op_rate_approx numeric dtype (실제 float64)
  [PASS] industry.firms_in numeric dtype (실제 float64)
  [PASS] industry.firms_op numeric dtype (실제 float64)
  [PASS] total.production_total numeric dtype (실제 float64)
  [PASS] total.employment_total numeric dtype (실제 float64)
  [PASS

## 5. 분석기간 확정

`validate_master()`가 산출한 `yoy_valid_quarters`(10개 업종 모두 4분면 산출 가능한 분기)를 근거로
본분석기간을 확정한다.

- 2018Q4·2020Q3 업종 재분류 이후 최소 4분기 버퍼가 필요해 2021Q3 이후부터 YoY 비교가 가능해지지만,
  2021년 구간은 섬유의복 업종의 2020년 생산·고용이 0이라 YoY가 inf가 되어 국면 분류가 불가능하다.
- 2023Q4 업종별 생산이 X(비공개)라 2023Q4·2024Q4 production YoY는 전 업종 계산 불가하다(정상적 구조 결측).
- 결과적으로 10개 업종 모두 안정적으로 국면 산출이 가능한 첫 분기는 **2022Q1**이다.

따라서 **참고기간 2018Q1~2026Q2 / 본분석기간 2022Q1~2026Q2**로 확정한다
(사용자 승인 완료, `src/build_kicox_analysis_panel.py`의 `START`/`END` 상수와 일치).

In [5]:
print('YoY 4분면 산출 가능 분기 수:', len(latest['yoy_valid_quarters']))
print('가능:', latest['yoy_valid_quarters'])
main_start, main_end = '2022Q1', '2026Q2'
in_main = [q for q in latest['yoy_valid_quarters'] if main_start <= q <= main_end]
print(f'본분석기간({main_start}~{main_end}) 내 유효 분기 수:', len(in_main))
assert min(latest['yoy_valid_quarters']) == main_start, '실측 유효분기 시작과 본분석기간 시작이 다릅니다'

YoY 4분면 산출 가능 분기 수: 16
가능: ['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2024Q1', '2024Q2', '2024Q3', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']
본분석기간(2022Q1~2026Q2) 내 유효 분기 수: 16


## 6~11. YoY·보조변수, 국면(S1~S4)·sensitivity(±0.5/1/2%)·run·transition·recent4 변수 생성

`src/build_kicox_analysis_panel.py`가 한 번에 처리한다.
- `prepare()`: production/employment/firms YoY(업종별 groupby 내부 lag4만 사용), 가동률 %p 변화,
  production_share/employment_share/active_firm_ratio, 재분류 위험(2018Q4/2020Q3) 플래그.
- `classify(threshold=0)`: 본분석 국면. **정확히 0이면 N**, 결측/계산불가는 **INVALID**(N과 다름).
- `classify(threshold=0.5/1/2)`: sensitivity 전용. 본분석 state와 완전히 분리된 별도 패널.
- `sequences()`: run_id/run_length(같은 업종+연속분기만, INVALID를 만나면 끊김)와
  transition(같은 업종+실제 연속분기일 때만 유효), recent4_S1~S4_count.

In [6]:
panel_res = subprocess.run([sys.executable, str(ROOT/'src'/'build_kicox_analysis_panel.py')],
                           capture_output=True, text=True, encoding='utf-8', errors='replace',
                           env=_env, cwd=ROOT)
print('build_kicox_analysis_panel.py exit code:', panel_res.returncode)
print(panel_res.stdout[-3000:])
if panel_res.returncode != 0:
    print('STDERR:', panel_res.stderr[-3000:])
assert panel_res.returncode == 0

build_kicox_analysis_panel.py exit code: 0
{
  "pipeline": "offline_reproducible",
  "main_period": [
    "2022Q1",
    "2026Q2"
  ],
  "reference_period": [
    "2018Q1",
    "2026Q2"
  ],
  "rows": 180,
  "columns": 123,
  "industries": 10,
  "valid_state": 160,
  "invalid_state": 20,
  "states": {
    "S1": 54,
    "S4": 46,
    "S2": 43,
    "INVALID": 20,
    "S3": 13,
    "N": 4
  },
  "exact_zero": 4,
  "valid_transition4": 126,
  "valid_transition5": 130,
  "changed_transition4": 40,
  "review_rows": 52,
  "missing": {
    "production": 10,
    "employment": 0,
    "production_yoy": 20,
    "employment_yoy": 0,
    "op_rate_official": 90,
    "op_rate_approx": 90
  },
  "run_total_length_distribution": {
    "1": 31,
    "2": 14,
    "3": 14,
    "4": 7,
    "6": 1,
    "7": 3
  },
  "near_zero_distribution": {
    "production": {
      "0": 0,
      "0.5": 2,
      "1": 5,
      "2": 12
    },
    "employment": {
      "0": 4,
      "0.5": 9,
      "1": 18,
      "2": 37
    }

In [7]:
state_main = pd.read_csv(DIR_PROC / 'changwon_state_panel.csv')
state_ref = pd.read_csv(DIR_PROC / 'changwon_state_reference_panel.csv')
state_sens = pd.read_csv(DIR_PROC / 'changwon_state_sensitivity_panel.csv')
quality = json.load(open(DIR_PROC / 'quality_report.json', encoding='utf-8'))

print('본분석 패널  :', state_main.shape, '(threshold=0)')
print('참고기간 패널:', state_ref.shape)
print('sensitivity  :', state_sens.shape, '(threshold=0.5/1/2)')
print()
print('본분석 국면 분포(threshold=0):')
print(state_main.state.value_counts().to_string())
print()
print('N과 INVALID는 서로 다른 상태다:')
print(state_main.loc[state_main.state=='N', ['quarter','industry','production_yoy','employment_yoy']].head(3).to_string(index=False))
print(state_main.loc[state_main.state=='INVALID', 'quarter'].unique())

본분석 패널  : (180, 123) (threshold=0)
참고기간 패널: (340, 123)
sensitivity  : (540, 123) (threshold=0.5/1/2)

본분석 국면 분포(threshold=0):
state
S1         54
S4         46
S2         43
INVALID    20
S3         13
N           4

N과 INVALID는 서로 다른 상태다:
quarter industry  production_yoy  employment_yoy
 2023Q3     섬유의복      166.498316             0.0
 2025Q4     섬유의복      -18.659812             0.0
 2026Q1     섬유의복       -9.239753             0.0


<StringArray>
['2023Q4', '2024Q4']
Length: 2, dtype: str


In [8]:
print('sensitivity(threshold별 국면 분포):')
for h, g in state_sens.groupby('threshold'):
    print(f'  threshold={h}: ' + str(g.state.value_counts().to_dict()))
print()
print('run_total_length 분포:')
print(state_main.dropna(subset=['run_id']).drop_duplicates('run_id').run_total_length.value_counts().sort_index().to_string())
print()
print('4상태 인접 전환 수(valid_transition):', int(state_main.valid_transition.sum()))
print('N 포함 인접 전환 수(valid_transition5):', int(state_main.valid_transition5.sum()))
print('recent4_S1_count 예시:')
print(state_main[['quarter','industry','recent4_S1_count','recent4_S2_count','recent4_S3_count','recent4_S4_count']].tail(5).to_string(index=False))

sensitivity(threshold별 국면 분포):
  threshold=0.5: {'S1': 51, 'S4': 46, 'S2': 41, 'INVALID': 20, 'N': 11, 'S3': 11}
  threshold=1.0: {'S1': 50, 'S4': 43, 'S2': 33, 'N': 23, 'INVALID': 20, 'S3': 11}
  threshold=2.0: {'N': 47, 'S1': 41, 'S4': 36, 'S2': 29, 'INVALID': 20, 'S3': 7}

run_total_length 분포:
run_total_length
1.0    31
2.0    14
3.0    14
4.0     7
6.0     1
7.0     3

4상태 인접 전환 수(valid_transition): 126
N 포함 인접 전환 수(valid_transition5): 130
recent4_S1_count 예시:
quarter industry  recent4_S1_count  recent4_S2_count  recent4_S3_count  recent4_S4_count
 2025Q2       철강               1.0               0.0               2.0               0.0
 2025Q3       철강               1.0               1.0               1.0               0.0
 2025Q4       철강               1.0               2.0               1.0               0.0
 2026Q1       철강               1.0               3.0               0.0               0.0
 2026Q2       철강               0.0               4.0               0.0               0

## 12. PPI validation 전처리

PPI 업종별 매핑은 아직 사용자 승인 전이므로(보류), 이번 파이프라인은 **총지수 기준의 제한적
민감도만** 준비한다. `data/processed/ppi/ppi_industry_mapping_candidates.csv`는 후보표일 뿐이며
`confirmed=False`로 저장되고, 업종별 실질생산 계산에는 쓰이지 않는다.

## 13. EIS validation 전처리

EIS는 `data/raw/eis/eis_changwon_insured_2022M03_2026M06.csv`(2022Q1~2026Q2, 18개 분기,
분기말월 stock)를 원자료로 사용한다. `build_validation_panels.py`가 저장 전에 다음을 강제 검증한다
(하나라도 실패하면 예외 발생):

- 기간이 정확히 2022Q1~2026Q2(18개 분기)와 일치
- `quarter` 중복 없음
- 주요 변수(quarter, month, changwon_total_all_industry, changwon_manufacturing, 5개 구
  제조업) 결측 없음
- `quarter`와 `month`의 분기말월 대응 일치(Q1=03월, Q2=06월, Q3=09월, Q4=12월)
- 5개 구(의창/성산/마산합포/마산회원/진해) 제조업 합계 == `changwon_manufacturing` (전 분기)
- 핵심 수치 컬럼이 전부 숫자형이며 음수 없음

KICOX 고용과 모집단이 달라 비율/점유율/보정값은 만들지 않는다. 검증 통과 후 KICOX
`employment_total`을 quarter 기준으로만 옆에 붙이고, 검증용으로 EIS 제조업(5개구 합) 피보험자수의
전년동분기대비 증감률(`eis_manufacturing_yoy_pct`)만 파생변수로 추가한다 — 이 값은 EIS 내부
검증용일 뿐 KICOX 핵심 분석 구조(state/run/transition)에는 사용하지 않는다.

In [9]:
val_res = subprocess.run([sys.executable, str(ROOT/'src'/'build_validation_panels.py')],
                         capture_output=True, text=True, encoding='utf-8', errors='replace',
                         env=_env, cwd=ROOT)
print(val_res.stdout)
if val_res.returncode != 0:
    print('STDERR:', val_res.stderr[-2000:])
assert val_res.returncode == 0

ppi_panel = pd.read_csv(ROOT / 'data' / 'processed' / 'ppi' / 'ppi_validation_panel.csv')
ppi_mapping = pd.read_csv(ROOT / 'data' / 'processed' / 'ppi' / 'ppi_industry_mapping_candidates.csv')
eis_panel = pd.read_csv(ROOT / 'data' / 'processed' / 'eis' / 'eis_validation_panel.csv')

print('PPI validation panel:', ppi_panel.shape, ppi_panel.quarter.min(), '~', ppi_panel.quarter.max())
print('PPI 매핑 후보 confirmed 값:', ppi_mapping.confirmed.unique(), '(전부 False 여야 함)')
assert (ppi_mapping.confirmed == False).all()

print('EIS validation panel:', eis_panel.shape, eis_panel.quarter.min(), '~', eis_panel.quarter.max())
assert len(eis_panel) == 18 and eis_panel.quarter.min() == '2022Q1' and eis_panel.quarter.max() == '2026Q2'
assert eis_panel.quarter.is_unique
gu_cols = ['uichang_mfg', 'seongsan_mfg', 'masanhappo_mfg', 'masanhoewon_mfg', 'jinhae_mfg']
assert (eis_panel[gu_cols].sum(axis=1) == eis_panel.changwon_manufacturing).all()
print('5개 구 제조업 합 == changwon_manufacturing: 전 분기 일치 확인')
print()
print('eis_manufacturing_yoy_pct (검증용 파생변수, KICOX 구조 미사용) 앞/뒤 4개 분기:')
print(eis_panel[['quarter', 'changwon_manufacturing', 'eis_manufacturing_yoy_pct']].head(4).to_string(index=False))
print(eis_panel[['quarter', 'changwon_manufacturing', 'eis_manufacturing_yoy_pct']].tail(4).to_string(index=False))

[PPI] ppi_validation_panel.csv 저장 (14행, 2023Q1~2026Q2)
[PPI] ppi_industry_mapping_candidates.csv 저장 (10건, 전부 confirmed=False — 보류)
[EIS] eis_validation_panel.csv 저장 (18행, 2022Q1~2026Q2)
[EIS] 검증 통과: 기간 18개 분기, quarter 중복 없음, 주요 변수 결측 없음, quarter-month 분기말월 대응 일치, 5개 구 제조업 합계 == changwon_manufacturing, 전 컬럼 숫자형·음수 없음

PPI validation panel: (14, 6) 2023Q1 ~ 2026Q2
PPI 매핑 후보 confirmed 값: [False] (전부 False 여야 함)
EIS validation panel: (18, 12) 2022Q1 ~ 2026Q2
5개 구 제조업 합 == changwon_manufacturing: 전 분기 일치 확인

eis_manufacturing_yoy_pct (검증용 파생변수, KICOX 구조 미사용) 앞/뒤 4개 분기:
quarter  changwon_manufacturing  eis_manufacturing_yoy_pct
 2022Q1                  110281                        NaN
 2022Q2                  110525                        NaN
 2022Q3                  110156                        NaN
 2022Q4                  110008                        NaN
quarter  changwon_manufacturing  eis_manufacturing_yoy_pct
 2025Q3                  113976                   0.110672
 2025Q4         

## 14. Final QA & Save

- `tests/test_pipeline_rules.py`로 핵심 규칙(업종 경계 lag4, N≠INVALID, transition 유효성,
  run reset, X 보존, revision 반영, raw 쓰기 금지)을 검증한다.
- `_previous/`와의 diff는 이미 3~4절에서 `build_changwon_master.py` 실행 로그에 포함되어 있다.
- data/raw/ 트리가 이 노트북 실행 전후 변경되지 않았는지 마지막에 다시 확인한다.

In [10]:
test_res = subprocess.run([sys.executable, '-m', 'pytest', str(ROOT/'tests'/'test_pipeline_rules.py'), '-v'],
                          capture_output=True, text=True, encoding='utf-8', errors='replace',
                          env=_env, cwd=ROOT)
print(test_res.stdout[-4000:])
if test_res.returncode != 0:
    print('STDERR:', test_res.stderr[-2000:])
assert test_res.returncode == 0

============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\jiwoo\Desktop\GitHub\Changwon-Industry-Employment-Monitor\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\jiwoo\Desktop\GitHub\Changwon-Industry-Employment-Monitor
plugins: anyio-4.15.0
collecting ... collected 12 items

tests/test_pipeline_rules.py::test_yoy_independent_recalculation_matches_within_industry PASSED [  8%]
tests/test_pipeline_rules.py::test_lag4_never_crosses_industry_boundary PASSED [ 16%]
tests/test_pipeline_rules.py::test_exact_zero_yoy_is_state_n PASSED      [ 25%]
tests/test_pipeline_rules.py::test_missing_quarters_are_invalid_not_n PASSED [ 33%]
tests/test_pipeline_rules.py::test_n_and_invalid_are_mutually_exclusive PASSED [ 41%]
tests/test_pipeline_rules.py::test_transition_only_within_same_industry_and_adjacent_quarter PASSED [ 50%]
tests/test_pipeline_rules.py::test_no_transition_across_invali

In [11]:
_raw_snapshot_after = bcm._raw_mtimes()
assert _raw_snapshot_before == _raw_snapshot_after, 'data/raw/ 가 노트북 실행 중 변경되었습니다'
print('[가드] data/raw/ 트리 변경 없음 확인 (노트북 전체 실행 기준)')

print()
print('=== 최종 산출물 ===')
for p in sorted((ROOT/'data'/'processed').rglob('*.csv')):
    if '_previous' in p.parts:
        continue
    print(' ', p.relative_to(ROOT))

[가드] data/raw/ 트리 변경 없음 확인 (노트북 전체 실행 기준)

=== 최종 산출물 ===
  data\processed\eis\eis_validation_panel.csv
  data\processed\kicox\changwon_industry_master.csv
  data\processed\kicox\changwon_state_panel.csv
  data\processed\kicox\changwon_state_reference_panel.csv
  data\processed\kicox\changwon_state_sensitivity_panel.csv
  data\processed\kicox\changwon_total_master.csv
  data\processed\ppi\ppi_industry_mapping_candidates.csv
  data\processed\ppi\ppi_validation_panel.csv


## 해석 주의

- 생산과 고용의 관계는 상관이며 인과가 아니다.
- 업종 재분류 시점(2018Q4, 2020Q3)을 가로지르는 장기 비교는 구조 변화로 해석하지 않는다.
- 2023Q4·2024Q4 production YoY 결측은 데이터 오류가 아니라 정상적인 구조적 결측이다.
- 업종합 employment != 산단 전체 employment 는 정상 특성이다(과거 확인범위 0.46~4.05%, 중앙값 1.27%).
- PPI 업종별 매핑은 미확정이며, 이 파이프라인은 총지수 기준 제한적 민감도만 제공한다.
- EIS는 2022Q1~2026Q2 겹치는 구간에서만 KICOX와 나란히 비교하고, 비율/점유율/보정값은 계산하지 않는다.
- `eis_manufacturing_yoy_pct`는 EIS 제조업(5개구 합) 피보험자수만의 검증용 전년동분기대비
  증감률이며, KICOX production/employment YoY와는 별개 지표다(2022년은 lag4 기준값 부재로 결측).

다음 단계: `02_eda.ipynb`